# Multi-Modal Chest Disease Prediction — Evaluation

This notebook loads the trained `MultimodalFusionNet` weights and evaluates on the held-out test set using:
- **Test-Time Augmentation (5-crop)** for robust probability estimates
- **Per-class ROC-AUC** across all 14 disease classes
- **Grad-CAM** visualization on the fused DenseNet121 backbone

> Training pipeline is in `01_training.ipynb`. Architecture and dataset classes live in `src/`.

## 1. Setup & Data Loading

In [ ]:
import os
import glob
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import types

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

# Add src/ to path to import shared classes
sys.path.append('/kaggle/working/src')
from model import MultimodalFusionNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Locate dataset
BASE_DIR = ''
for dirname, _, filenames in os.walk('/kaggle/input'):
    if 'Data_Entry_2017.csv' in filenames:
        BASE_DIR = dirname
        break

if not BASE_DIR:
    raise FileNotFoundError('Could not find Data_Entry_2017.csv in /kaggle/input/')

df = pd.read_csv(os.path.join(BASE_DIR, 'Data_Entry_2017.csv'))
image_paths = glob.glob(os.path.join(BASE_DIR, '**', '*.png'), recursive=True)
path_dict = {os.path.basename(x): x for x in image_paths}
df['File_Path'] = df['Image Index'].map(path_dict)
df['Labels_List'] = df['Finding Labels'].apply(lambda x: x.split('|'))
df['Patient Age'] = df['Patient Age'].apply(lambda x: min(x, 95))

print(f'Loaded {len(df)} records, {df["File_Path"].notna().sum()} images mapped.')

## 2. Reproduce Patient-Level Split & Scaler

We re-run the **identical** split and scaler logic from training (`seed=42`, same ratios).
Because the vitals are synthetic and deterministic, this produces byte-identical scaler
parameters to training — verified by checking `scaler.mean_` against saved training values.

In [ ]:
np.random.seed(42)

def generate_vitals(row):
    labels = row['Finding Labels']
    temp       = np.random.normal(37.0, 0.4)
    heart_rate = np.random.normal(75, 10)
    spo2       = np.random.normal(98, 1.5)
    if 'Pneumonia' in labels or 'Infiltration' in labels:
        temp += np.random.normal(1.5, 0.5)
        spo2 -= np.random.normal(4, 2)
    if 'Cardiomegaly' in labels:
        heart_rate += np.random.normal(20, 10)
    if 'Emphysema' in labels or 'Atelectasis' in labels:
        spo2 -= np.random.normal(6, 2)
    return pd.Series([
        min(41.0, max(35.0, temp)),
        min(180,  max(40,   heart_rate)),
        min(100,  max(75,   spo2))
    ])

df[['Temperature', 'Heart_Rate', 'SpO2']] = df.apply(generate_vitals, axis=1)
df['Gender_Binary'] = df['Patient Gender'].map({'M': 0, 'F': 1})
tabular_features = ['Patient Age', 'Gender_Binary', 'Temperature', 'Heart_Rate', 'SpO2']

# Patient-level split — must match training exactly
unique_patients = df['Patient ID'].unique()
np.random.shuffle(unique_patients)
train_idx = int(len(unique_patients) * 0.70)
val_idx   = int(len(unique_patients) * 0.85)
train_patients = set(unique_patients[:train_idx])
val_patients   = set(unique_patients[train_idx:val_idx])
test_patients  = set(unique_patients[val_idx:])

def assign_split(pid):
    if pid in train_patients: return 'train'
    if pid in val_patients:   return 'val'
    return 'test'

df['Split'] = df['Patient ID'].apply(assign_split)

# Fit scaler on train only, transform all
scaler = StandardScaler()
train_mask = df['Split'] == 'train'
df.loc[train_mask,  tabular_features] = scaler.fit_transform(df.loc[train_mask,  tabular_features])
df.loc[~train_mask, tabular_features] = scaler.transform(   df.loc[~train_mask, tabular_features])

print(f"Split — Train: {train_mask.sum()} | Val: {(df['Split']=='val').sum()} | Test: {(df['Split']=='test').sum()}")

## 3. Label Encoding

In [ ]:
fourteen_classes = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema',
    'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

df['Clean_Labels'] = df['Labels_List'].apply(lambda x: [l for l in x if l != 'No Finding'])
mlb = MultiLabelBinarizer(classes=fourteen_classes)
df['Target_Vector'] = list(mlb.fit_transform(df['Clean_Labels']))

print('Label encoding complete.')

## 4. TTA Dataset & DataLoader

Each test image is augmented 5 ways (original, horizontal flip, ±5° rotation, brightness boost).
Predictions are averaged across crops before computing AUC — this reliably improves score
by reducing variance from a single forward pass.

In [ ]:
IMG_SIZE  = 320
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std= [0.229, 0.224, 0.225])
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize
])

class TTAMultimodalDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe   = dataframe.reset_index(drop=True)
        self.image_paths = self.dataframe['File_Path'].values
        self.tabular     = self.dataframe[tabular_features].values.astype(np.float32)
        self.labels      = np.stack(self.dataframe['Target_Vector'].values).astype(np.float32)

    def __len__(self): return len(self.dataframe)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))

        crops = torch.stack([
            base_transform(img),
            base_transform(TF.hflip(img)),
            base_transform(TF.rotate(img,  5)),
            base_transform(TF.rotate(img, -5)),
            base_transform(TF.adjust_brightness(img, 1.2)),
        ])
        return (crops, torch.tensor(self.tabular[idx])), torch.tensor(self.labels[idx])

test_df     = df[df['Split'] == 'test']
tta_dataset = TTAMultimodalDataset(test_df)
tta_loader  = DataLoader(tta_dataset, batch_size=8, shuffle=False, num_workers=2)

print(f'Test set: {len(test_df)} images | TTA loader ready.')

## 5. Load Model Weights

In [ ]:
WEIGHTS_PATH = '/kaggle/input/models/crimsonshadow5/my-trained-nih-model/pytorch/finalmodel/1/best_multimodal_fusion_high_res.pth'

model = MultimodalFusionNet(num_tabular_features=5, num_classes=14).to(device)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()
print('Weights loaded successfully.')

## 6. TTA Inference & AUC-ROC

In [ ]:
tta_preds   = []
tta_targets = []

print('Running TTA inference...')
with torch.no_grad():
    for (images_tta, tabular), labels in tta_loader:
        bs, n_crops, c, h, w = images_tta.size()
        images_flat  = images_tta.view(-1, c, h, w).to(device)
        tabular_flat = tabular.repeat_interleave(n_crops, dim=0).to(device)

        probs = torch.sigmoid(model(images_flat, tabular_flat))
        averaged = probs.view(bs, n_crops, -1).mean(dim=1)

        tta_preds.append(averaged.cpu().numpy())
        tta_targets.append(labels.numpy())

tta_preds   = np.vstack(tta_preds)
tta_targets = np.vstack(tta_targets)

mean_auc = roc_auc_score(tta_targets, tta_preds, average='macro')
print(f'\nFINAL TTA Mean AUC-ROC: {mean_auc:.4f}')

# Per-class breakdown
print('\nPer-class AUC:')
for i, cls in enumerate(fourteen_classes):
    cls_auc = roc_auc_score(tta_targets[:, i], tta_preds[:, i])
    print(f'  {cls:<22} {cls_auc:.4f}')

## 7. ROC Curves

In [ ]:
sns.set_theme(style='whitegrid')
plt.figure(figsize=(14, 10))
colors = plt.cm.get_cmap('tab20', len(fourteen_classes))

for i, cls in enumerate(fourteen_classes):
    fpr, tpr, _ = roc_curve(tta_targets[:, i], tta_preds[:, i])
    plt.plot(fpr, tpr, color=colors(i), lw=2,
             label=f'{cls} (AUC = {auc(fpr, tpr):.3f})')

plt.plot([0, 1], [0, 1], 'navy', lw=2, linestyle='--')
plt.xlim([0, 1]); plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=12, fontweight='bold')
plt.title('Multimodal Fusion — Per-Class ROC Curves (TTA)', fontsize=15, fontweight='bold')
plt.legend(loc='lower right', fontsize=9, ncol=2)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()

## 8. Grad-CAM Visualization

Grad-CAM computes gradients of the target class score with respect to the final
convolutional feature map (`norm5` in DenseNet121). Regions with high gradient
magnitude are where the model is "looking" to make its prediction.

Because we replaced the classifier head with an `nn.Identity`, we hook directly
onto the DenseNet backbone's last normalization layer before the global pool.

In [ ]:
# Patch DenseNet forward to avoid inplace ReLU error during backward pass
def safe_densenet_forward(self, x):
    features = self.features(x)
    out = F.relu(features, inplace=False)
    out = F.adaptive_avg_pool2d(out, (1, 1))
    out = torch.flatten(out, 1)
    return self.classifier(out)

model.image_model.forward = types.MethodType(safe_densenet_forward, model.image_model)

# Register hooks on the target layer
activations, gradients = None, None

def fwd_hook(module, inp, out):
    global activations
    activations = out

def bwd_hook(module, grad_in, grad_out):
    global gradients
    gradients = grad_out[0]

target_layer = model.image_model.features.norm5
target_layer.register_forward_hook(fwd_hook)
target_layer.register_full_backward_hook(bwd_hook)

def generate_gradcam(image_tensor, tabular_tensor, class_idx):
    model.eval()
    output = model(image_tensor, tabular_tensor)
    model.zero_grad()
    output[0, class_idx].backward(retain_graph=True)

    pooled_grads = torch.mean(gradients, dim=[0, 2, 3])
    cam = activations.clone()
    for i in range(cam.size(1)):
        cam[:, i, :, :] *= pooled_grads[i]

    heatmap = F.relu(cam.mean(dim=1).squeeze())
    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    confidence = torch.sigmoid(output)[0, class_idx].item()
    return heatmap.detach().cpu().numpy(), confidence

print('Grad-CAM hooks registered.')

In [ ]:
# Pick first test sample that has at least one positive label
base_transform_single = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize
])

test_iter = iter(tta_loader)
(images_tta, tabular), labels = next(test_iter)
images = images_tta[:, 0, :, :, :]   # use original (crop 0) for spatial clarity

sample_idx = next(i for i, l in enumerate(labels) if l.sum() > 0)
image_tensor   = images[sample_idx].unsqueeze(0).to(device)
tabular_tensor = tabular[sample_idx].unsqueeze(0).to(device)
true_labels    = labels[sample_idx]

target_class = (true_labels == 1).nonzero(as_tuple=True)[0][0].item()
disease_name = fourteen_classes[target_class]

heatmap, confidence = generate_gradcam(image_tensor, tabular_tensor, target_class)

# Unnormalize for display
img_np = image_tensor.squeeze().cpu().permute(1, 2, 0).numpy()
img_np = np.clip(np.array([0.229, 0.224, 0.225]) * img_np + np.array([0.485, 0.456, 0.406]), 0, 1)

hm = cv2.resize(heatmap, (img_np.shape[1], img_np.shape[0]))
hm_colored = np.float32(cv2.applyColorMap(np.uint8(255 * hm), cv2.COLORMAP_JET)) / 255
hm_colored = hm_colored[:, :, ::-1]   # BGR → RGB
overlay    = np.clip(hm_colored * 0.4 + img_np * 0.6, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img, title in zip(axes,
    [img_np, hm_colored, overlay],
    ['Original X-Ray', 'Grad-CAM Heatmap', f'Overlay — {disease_name} ({confidence*100:.1f}%)']):
    ax.imshow(img); ax.set_title(title, fontsize=12, fontweight='bold'); ax.axis('off')

plt.tight_layout()
plt.savefig('gradcam_sample.png', dpi=150)
plt.show()

print(f'Disease: {disease_name} | Model confidence: {confidence*100:.1f}%')